# Cluster library & pip install audit

Answers two questions about this workspace:

1. **Part 1** — Which clusters (interactive and job, any state) have libraries installed, of any type, and where is each library installed from? Uses the Clusters and Libraries REST APIs via the Databricks SDK.
2. **Part 2** — Every time a `pip install` command ran in a notebook, from the `system.access.audit` system table.

**Notes / prerequisites**
* Part 1 needs no special permissions beyond cluster visibility, and takes a few minutes to enumerate every cluster in the workspace.
* Part 2 requires read access to `system.access.audit` (`USE CATALOG system`, `USE SCHEMA system.access`, `SELECT`).
* Part 2's `commandText` value is only recorded when **verbose audit logging** is enabled at the account level; without it, notebook command events are logged without the command text and the query returns zero rows.

## Part 1 — clusters with installed libraries

Enumerates every cluster visible in the workspace (`clusters.list`, which includes interactive and job clusters in any state, including terminated) and joins it with tracked library install state (`all-cluster-statuses`, which covers every cluster that still exists — including all currently running ones; long-terminated clusters drop out of the libraries API).

Output: one row per installed library, with the library type, its spec, and the **install location** — the file path for `whl`/`jar`/`egg`/`requirements` files, or the package index URL for `pypi`/`maven`/`cran` (blank repo means the default public index).

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Every cluster visible in the workspace (interactive + job clusters, any state)
clusters = {c.cluster_id: c for c in w.clusters.list()}

# Library install state for all clusters that still have tracked library state
# (includes every currently running cluster)
statuses = w.libraries.all_cluster_statuses()

def describe_library(lib):
    """Return (library_type, spec, install_location) for any library type."""
    if getattr(lib, "pypi", None):
        return "pypi", lib.pypi.package, lib.pypi.repo or "default public PyPI index"
    if getattr(lib, "maven", None):
        return "maven", lib.maven.coordinates, lib.maven.repo or "default Maven Central"
    if getattr(lib, "cran", None):
        return "cran", lib.cran.package, lib.cran.repo or "default CRAN"
    for path_type in ("whl", "jar", "egg", "requirements"):
        p = getattr(lib, path_type, None)
        if p:
            return path_type, p.split("/")[-1], p
    return "other", None, None

rows = []
for cs in statuses:
    c = clusters.get(cs.cluster_id)
    for ls in (cs.library_statuses or []):
        lib_type, spec, location = describe_library(ls.library)
        rows.append((
            c.cluster_name if c else None,
            cs.cluster_id,
            (c.state.value if c else None),
            (c.cluster_source.value if c and c.cluster_source else None),
            lib_type,
            spec,
            location,
            ls.status.value,
            ls.is_library_for_all_clusters,
            ("; ".join(ls.messages or [])[:500]) or None,
        ))

schema = ("cluster_name string, cluster_id string, cluster_state string, cluster_source string, "
          "library_type string, library_spec string, install_location string, install_status string, "
          "all_clusters_library boolean, install_messages string")
libs_df = spark.createDataFrame(rows, schema)
display(libs_df.orderBy("cluster_state", "cluster_name"))

## Part 2 — every `pip install` run in a notebook

Queries `system.access.audit` for `runCommand` events (executed notebook commands) whose command text contains `pip install` — this covers `%pip install`, `!pip install`, and `%sh pip install` alike. It includes both interactive notebooks (`service_name = 'notebook'`) and notebook cells executed by job runs (`service_name = 'jobs'`).

The query was validated for syntax and semantics (including the `request_params` map access and NULL handling) on a schema-identical mock. To run it for real you need read access to `system.access.audit`, and `commandText` is only populated when verbose audit logging is enabled — if you get zero rows, verify with a raw look at a `runCommand` event first:

```sql
SELECT request_params['commandText']
FROM system.access.audit
WHERE action_name = 'runCommand' AND service_name = 'notebook'
ORDER BY event_time DESC
LIMIT 5
```

In [0]:
%sql
-- Every notebook command that ran a pip install, from the audit system table.
-- Covers interactive notebooks ('notebook') and notebook cells executed by jobs ('jobs').
-- Requires read access to system.access.audit and verbose audit logging for commandText.
SELECT
  event_date,
  event_time,
  service_name,
  user_identity.email AS user_email,
  identity_metadata.run_by AS run_by,
  request_params['notebookId'] AS notebook_id,
  request_params['commandId'] AS command_id,
  request_params['commandLanguage'] AS command_language,
  request_params['status'] AS status,
  request_params['executionTime'] AS execution_time_sec,
  request_params['commandText'] AS command_text
FROM system.access.audit
WHERE action_name = 'runCommand'
  AND service_name IN ('notebook', 'jobs')
  AND lower(request_params['commandText']) LIKE '%pip install%'
  -- Narrow to recent history if the audit log is large:
  -- AND event_date >= DATE '2026-01-01'
ORDER BY event_time DESC

## Part 3 — metastore allowlist: approved JARs, init scripts, and Maven coordinates

The metastore **artifact allowlist** controls which JAR files, init scripts, and Maven coordinates are permitted to run on compute with standard access mode (formerly shared). It is managed from Catalog Explorer (metastore → *Allowed JARs/Init Scripts*) or the REST API, and is **empty by default** — anything not on the list is rejected on standard-access-mode compute.

This cell reads all three allowlists (`INIT_SCRIPT`, `LIBRARY_JAR`, `LIBRARY_MAVEN`) and lists every approved artifact pattern with its match type, plus who last set the allowlist and when. Entries are `PREFIX_MATCH` patterns: a path prefix for init scripts and JARs (e.g. a Unity Catalog Volume or DBFS directory), or a Maven `groupId`, `groupId:artifactId`, or full `groupId:artifactId:version` coordinate.

Prerequisite: reading the allowlist requires the `MANAGE ALLOWLIST` privilege on the metastore. Without it you get PERMISSION DENIED — an environment prerequisite, not a code bug (same situation as Part 2's audit access). Zero rows with no error means the allowlist exists but has no approved entries.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import ArtifactType
from datetime import datetime, timezone

w = WorkspaceClient()

allowlist_types = [
    ("INIT_SCRIPT", ArtifactType.INIT_SCRIPT),
    ("LIBRARY_JAR", ArtifactType.LIBRARY_JAR),
    ("LIBRARY_MAVEN", ArtifactType.LIBRARY_MAVEN),
]

rows = []
notes = []
for label, at in allowlist_types:
    try:
        info = w.artifact_allowlists.get(artifact_type=at)
        matchers = info.artifact_matchers or []
        if not matchers:
            notes.append(f"{label}: allowlist present but EMPTY - no approved entries")
            continue
        set_at = (datetime.fromtimestamp(info.created_at / 1000, tz=timezone.utc).isoformat()
                  if info.created_at else None)
        for m in matchers:
            rows.append((
                label,
                m.artifact,
                getattr(m.match_type, "value", str(m.match_type)),
                info.created_by,
                set_at,
                info.metastore_id,
            ))
    except Exception as e:
        msg = str(e).replace("\n", " ")
        if "MANAGE ALLOWLIST" in msg or "403" in msg:
            notes.append(f"{label}: PERMISSION DENIED - reading the artifact allowlist requires MANAGE ALLOWLIST on the metastore")
        else:
            notes.append(f"{label}: {type(e).__name__}: {msg[:300]}")

for n in notes:
    print(n)

schema = ("artifact_type string, approved_pattern string, match_type string, "
          "allowlist_set_by string, allowlist_set_at_utc string, metastore_id string")
allowlist_df = spark.createDataFrame(rows, schema)
display(allowlist_df)